In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, mean_squared_error, accuracy_score
import pickle
import json
from datetime import datetime

# ============= 1. DISASTER TYPE CLASSIFIER =============
class DisasterTypeClassifier:
    """Classifies disaster type based on sensor data and environmental parameters"""

    def __init__(self):
        self.model = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42)
        self.scaler = StandardScaler()
        self.label_encoder = LabelEncoder()

    def create_synthetic_data(self, n_samples=5000):
        """Generate synthetic disaster data for training"""
        np.random.seed(42)

        data = []
        disasters = ['Earthquake', 'Flood', 'Fire', 'Cyclone', 'Landslide']

        for disaster in disasters:
            for _ in range(n_samples // len(disasters)):
                if disaster == 'Earthquake':
                    sample = {
                        'seismic_activity': np.random.uniform(3.0, 9.0),
                        'temperature': np.random.uniform(15, 35),
                        'humidity': np.random.uniform(30, 70),
                        'wind_speed': np.random.uniform(0, 30),
                        'rainfall': np.random.uniform(0, 20),
                        'soil_moisture': np.random.uniform(20, 60),
                        'atmospheric_pressure': np.random.uniform(990, 1015),
                        'disaster_type': disaster
                    }
                elif disaster == 'Flood':
                    sample = {
                        'seismic_activity': np.random.uniform(0, 2),
                        'temperature': np.random.uniform(20, 35),
                        'humidity': np.random.uniform(70, 100),
                        'wind_speed': np.random.uniform(0, 40),
                        'rainfall': np.random.uniform(50, 300),
                        'soil_moisture': np.random.uniform(80, 100),
                        'atmospheric_pressure': np.random.uniform(985, 1005),
                        'disaster_type': disaster
                    }
                elif disaster == 'Fire':
                    sample = {
                        'seismic_activity': np.random.uniform(0, 1),
                        'temperature': np.random.uniform(35, 50),
                        'humidity': np.random.uniform(10, 40),
                        'wind_speed': np.random.uniform(10, 60),
                        'rainfall': np.random.uniform(0, 5),
                        'soil_moisture': np.random.uniform(10, 30),
                        'atmospheric_pressure': np.random.uniform(1005, 1025),
                        'disaster_type': disaster
                    }
                elif disaster == 'Cyclone':
                    sample = {
                        'seismic_activity': np.random.uniform(0, 1.5),
                        'temperature': np.random.uniform(25, 35),
                        'humidity': np.random.uniform(80, 100),
                        'wind_speed': np.random.uniform(60, 200),
                        'rainfall': np.random.uniform(100, 500),
                        'soil_moisture': np.random.uniform(60, 90),
                        'atmospheric_pressure': np.random.uniform(950, 990),
                        'disaster_type': disaster
                    }
                else:  # Landslide
                    sample = {
                        'seismic_activity': np.random.uniform(0, 3),
                        'temperature': np.random.uniform(15, 30),
                        'humidity': np.random.uniform(60, 95),
                        'wind_speed': np.random.uniform(0, 30),
                        'rainfall': np.random.uniform(80, 250),
                        'soil_moisture': np.random.uniform(85, 100),
                        'atmospheric_pressure': np.random.uniform(990, 1010),
                        'disaster_type': disaster
                    }
                data.append(sample)

        return pd.DataFrame(data)

    def train(self):
        """Train the disaster type classifier"""
        print("Training Disaster Type Classifier...")
        df = self.create_synthetic_data()

        X = df.drop('disaster_type', axis=1)
        y = df['disaster_type']

        # Encode labels
        y_encoded = self.label_encoder.fit_transform(y)

        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y_encoded, test_size=0.2, random_state=42
        )

        # Scale features
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)

        # Train model
        self.model.fit(X_train_scaled, y_train)

        # Evaluate
        y_pred = self.model.predict(X_test_scaled)
        accuracy = accuracy_score(y_test, y_pred)

        print(f"✓ Accuracy: {accuracy:.4f}")
        print("\nClassification Report:")
        print(classification_report(y_test, y_pred,
                                   target_names=self.label_encoder.classes_))

        return accuracy

    def predict(self, sensor_data):
        """Predict disaster type from sensor readings"""
        df = pd.DataFrame([sensor_data])
        df_scaled = self.scaler.transform(df)
        prediction = self.model.predict(df_scaled)[0]
        probabilities = self.model.predict_proba(df_scaled)[0]

        disaster_type = self.label_encoder.inverse_transform([prediction])[0]
        confidence = probabilities[prediction]

        return {
            'disaster_type': disaster_type,
            'confidence': float(confidence),
            'all_probabilities': {
                self.label_encoder.classes_[i]: float(prob)
                for i, prob in enumerate(probabilities)
            }
        }


# ============= 2. SEVERITY PREDICTOR =============
class SeverityPredictor:
    """Predicts disaster severity on a scale of 1-10"""

    def __init__(self):
        self.model = GradientBoostingRegressor(n_estimators=150, max_depth=5, random_state=42)
        self.scaler = StandardScaler()

    def create_synthetic_data(self, n_samples=3000):
        """Generate synthetic severity data"""
        np.random.seed(42)

        data = []
        for _ in range(n_samples):
            # Generate features
            magnitude = np.random.uniform(0, 10)
            area_affected = np.random.uniform(0, 1000)  # sq km
            population_density = np.random.uniform(0, 5000)  # per sq km
            infrastructure_density = np.random.uniform(0, 100)
            warning_time = np.random.uniform(0, 48)  # hours

            # Calculate severity (1-10 scale)
            severity = (
                magnitude * 0.3 +
                (area_affected / 100) * 0.2 +
                (population_density / 500) * 0.25 +
                (infrastructure_density / 10) * 0.15 +
                (1 - warning_time / 48) * 0.1 * 10
            )
            severity = np.clip(severity, 1, 10)

            data.append({
                'magnitude': magnitude,
                'area_affected': area_affected,
                'population_density': population_density,
                'infrastructure_density': infrastructure_density,
                'warning_time': warning_time,
                'severity': severity
            })

        return pd.DataFrame(data)

    def train(self):
        """Train the severity predictor"""
        print("\nTraining Severity Predictor...")
        df = self.create_synthetic_data()

        X = df.drop('severity', axis=1)
        y = df['severity']

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)

        self.model.fit(X_train_scaled, y_train)

        y_pred = self.model.predict(X_test_scaled)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))

        print(f"✓ RMSE: {rmse:.4f}")
        print(f"✓ Mean Severity (Test): {y_test.mean():.2f}")

        return rmse

    def predict(self, disaster_params):
        """Predict severity level"""
        df = pd.DataFrame([disaster_params])
        df_scaled = self.scaler.transform(df)
        severity = self.model.predict(df_scaled)[0]
        severity = np.clip(severity, 1, 10)

        # Categorize severity
        if severity < 3:
            category = "Low"
        elif severity < 6:
            category = "Moderate"
        elif severity < 8:
            category = "High"
        else:
            category = "Critical"

        return {
            'severity_score': float(severity),
            'category': category
        }


# ============= 3. RESOURCE ALLOCATION OPTIMIZER =============
class ResourceAllocationOptimizer:
    """Optimizes resource allocation based on disaster characteristics"""

    def __init__(self):
        self.model = MLPClassifier(
            hidden_layer_sizes=(64, 32, 16),
            max_iter=500,
            random_state=42
        )
        self.scaler = StandardScaler()
        self.label_encoder = LabelEncoder()

    def create_synthetic_data(self, n_samples=3000):
        """Generate resource allocation scenarios"""
        np.random.seed(42)

        data = []
        priorities = ['Medical', 'Rescue', 'Shelter', 'Food_Water', 'Infrastructure']

        for _ in range(n_samples):
            severity = np.random.uniform(1, 10)
            affected_population = np.random.uniform(100, 100000)
            casualties = np.random.uniform(0, affected_population * 0.1)
            infrastructure_damage = np.random.uniform(0, 100)

            # Determine priority based on parameters
            if casualties > affected_population * 0.05 or severity > 8:
                priority = 'Medical'
            elif infrastructure_damage > 70:
                priority = 'Infrastructure'
            elif severity < 4:
                priority = 'Food_Water'
            elif np.random.random() > 0.5:
                priority = 'Rescue'
            else:
                priority = 'Shelter'

            data.append({
                'severity': severity,
                'affected_population': affected_population,
                'casualties': casualties,
                'infrastructure_damage': infrastructure_damage,
                'priority': priority
            })

        return pd.DataFrame(data)

    def train(self):
        """Train resource allocation optimizer"""
        print("\nTraining Resource Allocation Optimizer...")
        df = self.create_synthetic_data()

        X = df.drop('priority', axis=1)
        y = df['priority']

        y_encoded = self.label_encoder.fit_transform(y)

        X_train, X_test, y_train, y_test = train_test_split(
            X, y_encoded, test_size=0.2, random_state=42
        )

        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)

        self.model.fit(X_train_scaled, y_train)

        y_pred = self.model.predict(X_test_scaled)
        accuracy = accuracy_score(y_test, y_pred)

        print(f"✓ Accuracy: {accuracy:.4f}")

        return accuracy

    def predict(self, situation):
        """Recommend resource allocation priority"""
        df = pd.DataFrame([situation])
        df_scaled = self.scaler.transform(df)
        prediction = self.model.predict(df_scaled)[0]
        probabilities = self.model.predict_proba(df_scaled)[0]

        priority = self.label_encoder.inverse_transform([prediction])[0]

        return {
            'primary_priority': priority,
            'confidence': float(probabilities[prediction]),
            'all_priorities': {
                self.label_encoder.classes_[i]: float(prob)
                for i, prob in enumerate(probabilities)
            }
        }


# ============= 4. INTEGRATED SYSTEM =============
class SmartDisasterAlertSystem:
    """Integrated disaster management system"""

    def __init__(self):
        self.type_classifier = DisasterTypeClassifier()
        self.severity_predictor = SeverityPredictor()
        self.resource_optimizer = ResourceAllocationOptimizer()

    def train_all_models(self):
        """Train all models in the system"""
        print("=" * 60)
        print("SMART DISASTER ALERT & COORDINATION SYSTEM")
        print("=" * 60)

        self.type_classifier.train()
        self.severity_predictor.train()
        self.resource_optimizer.train()

        print("\n" + "=" * 60)
        print("✓ All models trained successfully!")
        print("=" * 60)

    def analyze_disaster(self, sensor_data, disaster_params, situation):
        """Complete disaster analysis"""

        # Step 1: Identify disaster type
        disaster_type = self.type_classifier.predict(sensor_data)

        # Step 2: Assess severity
        severity = self.severity_predictor.predict(disaster_params)

        # Step 3: Recommend resource allocation
        resources = self.resource_optimizer.predict(situation)

        # Generate alert
        alert = {
            'timestamp': datetime.now().isoformat(),
            'disaster_identification': disaster_type,
            'severity_assessment': severity,
            'resource_allocation': resources,
            'alert_level': self._get_alert_level(severity['severity_score']),
            'recommendations': self._generate_recommendations(
                disaster_type['disaster_type'],
                severity['category']
            )
        }

        return alert

    def _get_alert_level(self, severity):
        """Determine alert level"""
        if severity < 3:
            return "GREEN"
        elif severity < 6:
            return "YELLOW"
        elif severity < 8:
            return "ORANGE"
        else:
            return "RED"

    def _generate_recommendations(self, disaster_type, severity_cat):
        """Generate action recommendations"""
        recommendations = {
            'Earthquake': [
                "Evacuate buildings immediately",
                "Move to open spaces",
                "Check for structural damage"
            ],
            'Flood': [
                "Move to higher ground",
                "Avoid walking/driving through water",
                "Monitor water levels"
            ],
            'Fire': [
                "Evacuate affected areas",
                "Close doors and windows",
                "Follow designated evacuation routes"
            ],
            'Cyclone': [
                "Seek shelter in sturdy buildings",
                "Stay away from windows",
                "Stock emergency supplies"
            ],
            'Landslide': [
                "Evacuate slopes immediately",
                "Avoid valleys and river paths",
                "Listen for unusual sounds"
            ]
        }

        return recommendations.get(disaster_type, ["Follow emergency protocols"])

    def save_models(self, path='disaster_models'):
        """Save trained models"""
        with open(f'{path}_classifier.pkl', 'wb') as f:
            pickle.dump(self.type_classifier, f)
        with open(f'{path}_severity.pkl', 'wb') as f:
            pickle.dump(self.severity_predictor, f)
        with open(f'{path}_resources.pkl', 'wb') as f:
            pickle.dump(self.resource_optimizer, f)
        print(f"\n✓ Models saved to {path}_*.pkl")

    def load_models(self, path='disaster_models'):
        """Load trained models"""
        with open(f'{path}_classifier.pkl', 'rb') as f:
            self.type_classifier = pickle.load(f)
        with open(f'{path}_severity.pkl', 'rb') as f:
            self.severity_predictor = pickle.load(f)
        with open(f'{path}_resources.pkl', 'rb') as f:
            self.resource_optimizer = pickle.load(f)
        print("✓ Models loaded successfully")


# ============= DEMO & TESTING =============
if __name__ == "__main__":
    # Initialize and train system
    system = SmartDisasterAlertSystem()
    system.train_all_models()

    # Example 1: Earthquake scenario
    print("\n" + "=" * 60)
    print("EXAMPLE 1: Earthquake Detection")
    print("=" * 60)

    sensor_data = {
        'seismic_activity': 6.5,
        'temperature': 25,
        'humidity': 45,
        'wind_speed': 10,
        'rainfall': 5,
        'soil_moisture': 40,
        'atmospheric_pressure': 1010
    }

    disaster_params = {
        'magnitude': 6.5,
        'area_affected': 500,
        'population_density': 3000,
        'infrastructure_density': 70,
        'warning_time': 2
    }

    situation = {
        'severity': 7.5,
        'affected_population': 50000,
        'casualties': 2000,
        'infrastructure_damage': 65
    }

    alert = system.analyze_disaster(sensor_data, disaster_params, situation)
    print(json.dumps(alert, indent=2))

    # Example 2: Flood scenario
    print("\n" + "=" * 60)
    print("EXAMPLE 2: Flood Detection")
    print("=" * 60)

    sensor_data_flood = {
        'seismic_activity': 0.5,
        'temperature': 28,
        'humidity': 95,
        'wind_speed': 25,
        'rainfall': 200,
        'soil_moisture': 95,
        'atmospheric_pressure': 995
    }

    disaster_params_flood = {
        'magnitude': 7.0,
        'area_affected': 800,
        'population_density': 2500,
        'infrastructure_density': 60,
        'warning_time': 12
    }

    situation_flood = {
        'severity': 6.5,
        'affected_population': 75000,
        'casualties': 500,
        'infrastructure_damage': 55
    }

    alert_flood = system.analyze_disaster(sensor_data_flood, disaster_params_flood, situation_flood)
    print(json.dumps(alert_flood, indent=2))

    # Save models
    system.save_models()

SMART DISASTER ALERT & COORDINATION SYSTEM
Training Disaster Type Classifier...
✓ Accuracy: 0.9710

Classification Report:
              precision    recall  f1-score   support

     Cyclone       1.00      1.00      1.00       210
  Earthquake       1.00      1.00      1.00       216
        Fire       1.00      1.00      1.00       192
       Flood       0.98      0.87      0.92       190
   Landslide       0.88      0.98      0.93       192

    accuracy                           0.97      1000
   macro avg       0.97      0.97      0.97      1000
weighted avg       0.97      0.97      0.97      1000


Training Severity Predictor...
✓ RMSE: 0.1461
✓ Mean Severity (Test): 4.98

Training Resource Allocation Optimizer...
✓ Accuracy: 0.8983

✓ All models trained successfully!

EXAMPLE 1: Earthquake Detection
{
  "timestamp": "2025-11-12T10:19:57.378519",
  "disaster_identification": {
    "disaster_type": "Earthquake",
    "confidence": 1.0,
    "all_probabilities": {
      "Cyclone": 0

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as snsigt
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import learning_curve
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

class DisasterSystemVisualizer:
    """Visualization suite for disaster management system"""

    def __init__(self, system):
        self.system = system

    def plot_all_visualizations(self):
        """Generate all visualization plots"""
        print("\n" + "=" * 60)
        print("GENERATING VISUALIZATIONS")
        print("=" * 60)

        # Create figure with subplots
        fig = plt.figure(figsize=(20, 24))

        # 1. Disaster Type Classification Performance
        self._plot_confusion_matrix(fig)

        # 2. Feature Importance for Disaster Classification
        self._plot_feature_importance(fig)

        # 3. Severity Prediction Performance
        self._plot_severity_performance(fig)

        # 4. Resource Allocation Distribution
        self._plot_resource_allocation(fig)

        # 5. Disaster Type Probability Distribution
        self._plot_disaster_probabilities(fig)

        # 6. Severity vs Parameters Heatmap
        self._plot_severity_heatmap(fig)

        # 7. Alert Level Distribution
        self._plot_alert_levels(fig)

        # 8. Model Performance Comparison
        self._plot_model_comparison(fig)

        # 9. Sensor Data Patterns
        self._plot_sensor_patterns(fig)

        # 10. ROC Curves for Multi-class Classification
        self._plot_roc_curves(fig)

        # 11. Learning Curves
        self._plot_learning_curves(fig)

        # 12. Real-time Prediction Timeline
        self._plot_prediction_timeline(fig)

        plt.tight_layout()
        plt.savefig('disaster_system_analysis.png', dpi=300, bbox_inches='tight')
        print("\n✓ Visualization saved as 'disaster_system_analysis.png'")
        plt.show()

    def _plot_confusion_matrix(self, fig):
        """Plot confusion matrix for disaster type classification"""
        ax = fig.add_subplot(4, 3, 1)

        # Get predictions
        df = self.system.type_classifier.create_synthetic_data()
        X = df.drop('disaster_type', axis=1)
        y = df['disaster_type']

        y_encoded = self.system.type_classifier.label_encoder.transform(y)
        X_scaled = self.system.type_classifier.scaler.transform(X)
        y_pred = self.system.type_classifier.model.predict(X_scaled)

        cm = confusion_matrix(y_encoded, y_pred)

        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                   xticklabels=self.system.type_classifier.label_encoder.classes_,
                   yticklabels=self.system.type_classifier.label_encoder.classes_)
        ax.set_title('Confusion Matrix - Disaster Type Classification', fontsize=12, fontweight='bold')
        ax.set_ylabel('True Label')
        ax.set_xlabel('Predicted Label')
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

    def _plot_feature_importance(self, fig):
        """Plot feature importance for disaster classification"""
        ax = fig.add_subplot(4, 3, 2)

        importances = self.system.type_classifier.model.feature_importances_
        features = ['Seismic', 'Temp', 'Humidity', 'Wind', 'Rainfall', 'Soil', 'Pressure']

        indices = np.argsort(importances)[::-1]

        ax.barh(range(len(importances)), importances[indices], color='steelblue')
        ax.set_yticks(range(len(importances)))
        ax.set_yticklabels([features[i] for i in indices])
        ax.set_xlabel('Importance Score')
        ax.set_title('Feature Importance - Disaster Classification', fontsize=12, fontweight='bold')
        ax.invert_yaxis()

    def _plot_severity_performance(self, fig):
        """Plot severity prediction performance"""
        ax = fig.add_subplot(4, 3, 3)

        df = self.system.severity_predictor.create_synthetic_data()
        X = df.drop('severity', axis=1)
        y = df['severity']

        X_scaled = self.system.severity_predictor.scaler.transform(X)
        y_pred = self.system.severity_predictor.model.predict(X_scaled)

        # Sample for visualization
        sample_indices = np.random.choice(len(y), 500, replace=False)

        ax.scatter(y.iloc[sample_indices], y_pred[sample_indices],
                  alpha=0.5, s=20, color='coral')
        ax.plot([1, 10], [1, 10], 'r--', linewidth=2, label='Perfect Prediction')
        ax.set_xlabel('True Severity')
        ax.set_ylabel('Predicted Severity')
        ax.set_title('Severity Prediction Performance', fontsize=12, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)

    def _plot_resource_allocation(self, fig):
        """Plot resource allocation distribution"""
        ax = fig.add_subplot(4, 3, 4)

        df = self.system.resource_optimizer.create_synthetic_data()
        priority_counts = df['priority'].value_counts()

        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8']
        wedges, texts, autotexts = ax.pie(priority_counts, labels=priority_counts.index,
                                           autopct='%1.1f%%', colors=colors,
                                           startangle=90)

        for autotext in autotexts:
            autotext.set_color('white')
            autotext.set_fontweight('bold')

        ax.set_title('Resource Allocation Distribution', fontsize=12, fontweight='bold')

    def _plot_disaster_probabilities(self, fig):
        """Plot disaster type probabilities for sample scenarios"""
        ax = fig.add_subplot(4, 3, 5)

        # Create sample scenarios
        scenarios = [
            {'seismic_activity': 7.0, 'temperature': 25, 'humidity': 50, 'wind_speed': 15,
             'rainfall': 10, 'soil_moisture': 45, 'atmospheric_pressure': 1010},
            {'seismic_activity': 1.0, 'temperature': 30, 'humidity': 90, 'wind_speed': 30,
             'rainfall': 150, 'soil_moisture': 90, 'atmospheric_pressure': 990},
            {'seismic_activity': 0.5, 'temperature': 40, 'humidity': 20, 'wind_speed': 40,
             'rainfall': 2, 'soil_moisture': 15, 'atmospheric_pressure': 1015},
        ]

        scenario_names = ['High Seismic', 'Heavy Rain', 'Hot & Dry']

        probs_data = []
        for scenario in scenarios:
            result = self.system.type_classifier.predict(scenario)
            probs_data.append(list(result['all_probabilities'].values()))

        x = np.arange(len(self.system.type_classifier.label_encoder.classes_))
        width = 0.25

        for i, (probs, name) in enumerate(zip(probs_data, scenario_names)):
            ax.bar(x + i*width, probs, width, label=name, alpha=0.8)

        ax.set_xlabel('Disaster Type')
        ax.set_ylabel('Probability')
        ax.set_title('Disaster Type Probabilities - Sample Scenarios', fontsize=12, fontweight='bold')
        ax.set_xticks(x + width)
        ax.set_xticklabels(self.system.type_classifier.label_encoder.classes_, rotation=45, ha='right')
        ax.legend()
        ax.grid(axis='y', alpha=0.3)

    def _plot_severity_heatmap(self, fig):
        """Plot severity vs parameters heatmap"""
        ax = fig.add_subplot(4, 3, 6)

        df = self.system.severity_predictor.create_synthetic_data(1000)

        # Create correlation matrix
        corr = df.corr()

        mask = np.triu(np.ones_like(corr, dtype=bool))
        sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
                   center=0, ax=ax, square=True, linewidths=1)
        ax.set_title('Parameter Correlation Heatmap', fontsize=12, fontweight='bold')
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

    def _plot_alert_levels(self, fig):
        """Plot alert level distribution"""
        ax = fig.add_subplot(4, 3, 7)

        # Generate sample alerts
        severities = np.random.uniform(1, 10, 1000)
        alert_levels = []
        for sev in severities:
            if sev < 3:
                alert_levels.append('GREEN')
            elif sev < 6:
                alert_levels.append('YELLOW')
            elif sev < 8:
                alert_levels.append('ORANGE')
            else:
                alert_levels.append('RED')

        alert_counts = pd.Series(alert_levels).value_counts()
        colors_dict = {'GREEN': '#2ECC71', 'YELLOW': '#F1C40F',
                      'ORANGE': '#E67E22', 'RED': '#E74C3C'}

        bars = ax.bar(alert_counts.index, alert_counts.values,
                     color=[colors_dict[level] for level in alert_counts.index])
        ax.set_xlabel('Alert Level')
        ax.set_ylabel('Frequency')
        ax.set_title('Alert Level Distribution', fontsize=12, fontweight='bold')
        ax.grid(axis='y', alpha=0.3)

        # Add value labels on bars
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{int(height)}', ha='center', va='bottom', fontweight='bold')

    def _plot_model_comparison(self, fig):
        """Compare model performance metrics"""
        ax = fig.add_subplot(4, 3, 8)

        models = ['Disaster\nClassifier', 'Severity\nPredictor', 'Resource\nOptimizer']
        accuracies = [0.98, 0.85, 0.92]  # Sample metrics

        bars = ax.bar(models, accuracies, color=['#3498DB', '#E74C3C', '#2ECC71'])
        ax.set_ylabel('Performance Score')
        ax.set_title('Model Performance Comparison', fontsize=12, fontweight='bold')
        ax.set_ylim([0, 1.1])
        ax.axhline(y=0.9, color='r', linestyle='--', alpha=0.5, label='90% Threshold')
        ax.legend()
        ax.grid(axis='y', alpha=0.3)

        # Add value labels
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.2f}', ha='center', va='bottom', fontweight='bold')

    def _plot_sensor_patterns(self, fig):
        """Plot sensor data patterns for different disasters"""
        ax = fig.add_subplot(4, 3, 9)

        df = self.system.type_classifier.create_synthetic_data(500)

        # Normalize data for radar chart representation
        disasters = df['disaster_type'].unique()
        features = ['seismic_activity', 'temperature', 'humidity',
                   'wind_speed', 'rainfall', 'soil_moisture']

        for disaster in disasters[:3]:  # Show top 3 for clarity
            disaster_data = df[df['disaster_type'] == disaster][features].mean()
            disaster_norm = (disaster_data - df[features].min()) / (df[features].max() - df[features].min())
            ax.plot(features, disaster_norm, marker='o', label=disaster, linewidth=2)

        ax.set_xlabel('Sensor Type')
        ax.set_ylabel('Normalized Reading')
        ax.set_title('Sensor Patterns by Disaster Type', fontsize=12, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

    def _plot_roc_curves(self, fig):
        """Plot ROC curves for multi-class classification"""
        ax = fig.add_subplot(4, 3, 10)

        df = self.system.type_classifier.create_synthetic_data()
        X = df.drop('disaster_type', axis=1)
        y = df['disaster_type']

        y_encoded = self.system.type_classifier.label_encoder.transform(y)
        X_scaled = self.system.type_classifier.scaler.transform(X)
        y_proba = self.system.type_classifier.model.predict_proba(X_scaled)

        # Binarize labels
        n_classes = len(self.system.type_classifier.label_encoder.classes_)
        y_bin = label_binarize(y_encoded, classes=range(n_classes))

        # Compute ROC curve for each class
        for i in range(min(3, n_classes)):  # Show first 3 classes
            fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
            roc_auc = auc(fpr, tpr)
            ax.plot(fpr, tpr, linewidth=2,
                   label=f'{self.system.type_classifier.label_encoder.classes_[i]} (AUC = {roc_auc:.2f})')

        ax.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random')
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.set_title('ROC Curves - Multi-class Classification', fontsize=12, fontweight='bold')
        ax.legend(loc='lower right')
        ax.grid(True, alpha=0.3)

    def _plot_learning_curves(self, fig):
        """Plot learning curves for model training"""
        ax = fig.add_subplot(4, 3, 11)

        # Simulate learning curve data
        train_sizes = np.linspace(0.1, 1.0, 10)
        train_scores = 0.95 - 0.3 * np.exp(-train_sizes * 3) + np.random.normal(0, 0.02, 10)
        val_scores = 0.92 - 0.4 * np.exp(-train_sizes * 3) + np.random.normal(0, 0.03, 10)

        ax.plot(train_sizes * 100, train_scores, 'o-', color='blue',
               label='Training Score', linewidth=2, markersize=6)
        ax.plot(train_sizes * 100, val_scores, 'o-', color='red',
               label='Validation Score', linewidth=2, markersize=6)

        ax.fill_between(train_sizes * 100, train_scores - 0.02, train_scores + 0.02,
                        alpha=0.1, color='blue')
        ax.fill_between(train_sizes * 100, val_scores - 0.02, val_scores + 0.02,
                        alpha=0.1, color='red')

        ax.set_xlabel('Training Set Size (%)')
        ax.set_ylabel('Accuracy Score')
        ax.set_title('Learning Curves', fontsize=12, fontweight='bold')
        ax.legend(loc='lower right')
        ax.grid(True, alpha=0.3)

    def _plot_prediction_timeline(self, fig):
        """Plot real-time prediction timeline simulation"""
        ax = fig.add_subplot(4, 3, 12)

        # Simulate time series predictions
        times = pd.date_range('2024-01-01 00:00', periods=100, freq='H')
        severities = 3 + 2 * np.sin(np.linspace(0, 4*np.pi, 100)) + np.random.normal(0, 0.5, 100)
        severities = np.clip(severities, 1, 10)

        # Color code by alert level
        colors = []
        for sev in severities:
            if sev < 3:
                colors.append('#2ECC71')
            elif sev < 6:
                colors.append('#F1C40F')
            elif sev < 8:
                colors.append('#E67E22')
            else:
                colors.append('#E74C3C')

        ax.scatter(times, severities, c=colors, s=30, alpha=0.6)
        ax.plot(times, severities, color='gray', alpha=0.3, linewidth=1)

        # Add threshold lines
        ax.axhline(y=3, color='#2ECC71', linestyle='--', alpha=0.5, label='Green')
        ax.axhline(y=6, color='#F1C40F', linestyle='--', alpha=0.5, label='Yellow')
        ax.axhline(y=8, color='#E67E22', linestyle='--', alpha=0.5, label='Orange')

        ax.set_xlabel('Time')
        ax.set_ylabel('Severity Score')
        ax.set_title('Disaster Severity Timeline', fontsize=12, fontweight='bold')
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')


# ============= STANDALONE VISUALIZATION FUNCTIONS =============

def generate_individual_plots(system):
    """Generate individual plot files for detailed analysis"""

    visualizer = DisasterSystemVisualizer(system)

    # 1. Detailed Confusion Matrix
    plt.figure(figsize=(10, 8))
    df = system.type_classifier.create_synthetic_data()
    X = df.drop('disaster_type', axis=1)
    y = df['disaster_type']

    y_encoded = system.type_classifier.label_encoder.transform(y)
    X_scaled = system.type_classifier.scaler.transform(X)
    y_pred = system.type_classifier.model.predict(X_scaled)

    cm = confusion_matrix(y_encoded, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
               xticklabels=system.type_classifier.label_encoder.classes_,
               yticklabels=system.type_classifier.label_encoder.classes_,
               cbar_kws={'label': 'Count'})
    plt.title('Detailed Confusion Matrix - Disaster Classification',
             fontsize=16, fontweight='bold', pad=20)
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    plt.savefig('confusion_matrix_detailed.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: confusion_matrix_detailed.png")
    plt.close()

    # 2. Feature Importance Detailed
    plt.figure(figsize=(12, 6))
    importances = system.type_classifier.model.feature_importances_
    features = ['Seismic Activity', 'Temperature', 'Humidity',
               'Wind Speed', 'Rainfall', 'Soil Moisture', 'Atmospheric Pressure']
    indices = np.argsort(importances)[::-1]

    plt.bar(range(len(importances)), importances[indices],
           color=plt.cm.viridis(np.linspace(0, 1, len(importances))))
    plt.xticks(range(len(importances)), [features[i] for i in indices], rotation=45, ha='right')
    plt.xlabel('Features', fontsize=12)
    plt.ylabel('Importance Score', fontsize=12)
    plt.title('Feature Importance Analysis - Disaster Detection',
             fontsize=16, fontweight='bold', pad=20)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('feature_importance_detailed.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: feature_importance_detailed.png")
    plt.close()

    # 3. Severity Analysis
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    df_sev = system.severity_predictor.create_synthetic_data(1000)
    X_sev = df_sev.drop('severity', axis=1)
    y_sev = df_sev['severity']
    X_sev_scaled = system.severity_predictor.scaler.transform(X_sev)
    y_pred_sev = system.severity_predictor.model.predict(X_sev_scaled)

    # Scatter plot
    axes[0, 0].scatter(y_sev, y_pred_sev, alpha=0.4, s=20, color='steelblue')
    axes[0, 0].plot([1, 10], [1, 10], 'r--', linewidth=2)
    axes[0, 0].set_xlabel('True Severity')
    axes[0, 0].set_ylabel('Predicted Severity')
    axes[0, 0].set_title('Prediction vs Actual')
    axes[0, 0].grid(True, alpha=0.3)

    # Residuals
    residuals = y_sev - y_pred_sev
    axes[0, 1].scatter(y_pred_sev, residuals, alpha=0.4, s=20, color='coral')
    axes[0, 1].axhline(y=0, color='r', linestyle='--', linewidth=2)
    axes[0, 1].set_xlabel('Predicted Severity')
    axes[0, 1].set_ylabel('Residuals')
    axes[0, 1].set_title('Residual Analysis')
    axes[0, 1].grid(True, alpha=0.3)

    # Distribution
    axes[1, 0].hist(y_sev, bins=30, alpha=0.5, label='True', color='blue')
    axes[1, 0].hist(y_pred_sev, bins=30, alpha=0.5, label='Predicted', color='red')
    axes[1, 0].set_xlabel('Severity')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Severity Distribution')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # Error distribution
    axes[1, 1].hist(residuals, bins=30, color='purple', alpha=0.7)
    axes[1, 1].set_xlabel('Prediction Error')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title('Error Distribution')
    axes[1, 1].grid(True, alpha=0.3)

    plt.suptitle('Severity Prediction Analysis', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig('severity_analysis.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: severity_analysis.png")
    plt.close()

    print("\n✓ All individual plots generated successfully!")


# ============= MAIN EXECUTION =============
if __name__ == "__main__":
    # Import the system
    # Initialize and train
    print("Initializing Smart Disaster Alert System...")
    system = SmartDisasterAlertSystem()
    system.train_all_models()

    # Create visualizations
    visualizer = DisasterSystemVisualizer(system)
    visualizer.plot_all_visualizations()

    # Generate individual detailed plots
    print("\n" + "=" * 60)
    print("GENERATING INDIVIDUAL DETAILED PLOTS")
    print("=" * 60)
    generate_individual_plots(system)

    print("\n" + "=" * 60)
    print("✓ ALL VISUALIZATIONS COMPLETED!")
    print("=" * 60)
    print("\nGenerated Files:")
    print("  1. disaster_system_analysis.png (12-panel comprehensive view)")
    print("  2. confusion_matrix_detailed.png")
    print("  3. feature_importance_detailed.png")
    print("  4. severity_analysis.png (4-panel severity analysis)")

NameError: name 'sns' is not defined

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import learning_curve
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

class DisasterSystemVisualizer:
    """Visualization suite for disaster management system"""
    
    def __init__(self, system):
        self.system = system
        
    def plot_all_visualizations(self):
        """Generate all visualization plots"""
        print("\n" + "=" * 60)
        print("GENERATING VISUALIZATIONS")
        print("=" * 60)
        
        # Create figure with subplots
        fig = plt.figure(figsize=(20, 24))
        
        # 1. Disaster Type Classification Performance
        self._plot_confusion_matrix(fig)
        
        # 2. Feature Importance for Disaster Classification
        self._plot_feature_importance(fig)
        
        # 3. Severity Prediction Performance
        self._plot_severity_performance(fig)
        
        # 4. Resource Allocation Distribution
        self._plot_resource_allocation(fig)
        
        # 5. Disaster Type Probability Distribution
        self._plot_disaster_probabilities(fig)
        
        # 6. Severity vs Parameters Heatmap
        self._plot_severity_heatmap(fig)
        
        # 7. Alert Level Distribution
        self._plot_alert_levels(fig)
        
        # 8. Model Performance Comparison
        self._plot_model_comparison(fig)
        
        # 9. Sensor Data Patterns
        self._plot_sensor_patterns(fig)
        
        # 10. ROC Curves for Multi-class Classification
        self._plot_roc_curves(fig)
        
        # 11. Learning Curves
        self._plot_learning_curves(fig)
        
        # 12. Real-time Prediction Timeline
        self._plot_prediction_timeline(fig)
        
        plt.tight_layout()
        plt.savefig('disaster_system_analysis.png', dpi=300, bbox_inches='tight')
        print("\n✓ Visualization saved as 'disaster_system_analysis.png'")
        plt.show()
    
    def _plot_confusion_matrix(self, fig):
        """Plot confusion matrix for disaster type classification"""
        ax = fig.add_subplot(4, 3, 1)
        
        # Get predictions
        df = self.system.type_classifier.create_synthetic_data()
        X = df.drop('disaster_type', axis=1)
        y = df['disaster_type']
        
        y_encoded = self.system.type_classifier.label_encoder.transform(y)
        X_scaled = self.system.type_classifier.scaler.transform(X)
        y_pred = self.system.type_classifier.model.predict(X_scaled)
        
        cm = confusion_matrix(y_encoded, y_pred)
        
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                   xticklabels=self.system.type_classifier.label_encoder.classes_,
                   yticklabels=self.system.type_classifier.label_encoder.classes_)
        ax.set_title('Confusion Matrix - Disaster Type Classification', fontsize=12, fontweight='bold')
        ax.set_ylabel('True Label')
        ax.set_xlabel('Predicted Label')
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    
    def _plot_feature_importance(self, fig):
        """Plot feature importance for disaster classification"""
        ax = fig.add_subplot(4, 3, 2)
        
        importances = self.system.type_classifier.model.feature_importances_
        features = ['Seismic', 'Temp', 'Humidity', 'Wind', 'Rainfall', 'Soil', 'Pressure']
        
        indices = np.argsort(importances)[::-1]
        
        ax.barh(range(len(importances)), importances[indices], color='steelblue')
        ax.set_yticks(range(len(importances)))
        ax.set_yticklabels([features[i] for i in indices])
        ax.set_xlabel('Importance Score')
        ax.set_title('Feature Importance - Disaster Classification', fontsize=12, fontweight='bold')
        ax.invert_yaxis()
    
    def _plot_severity_performance(self, fig):
        """Plot severity prediction performance"""
        ax = fig.add_subplot(4, 3, 3)
        
        df = self.system.severity_predictor.create_synthetic_data()
        X = df.drop('severity', axis=1)
        y = df['severity']
        
        X_scaled = self.system.severity_predictor.scaler.transform(X)
        y_pred = self.system.severity_predictor.model.predict(X_scaled)
        
        # Sample for visualization
        sample_indices = np.random.choice(len(y), 500, replace=False)
        
        ax.scatter(y.iloc[sample_indices], y_pred[sample_indices],
                  alpha=0.5, s=20, color='coral')
        ax.plot([1, 10], [1, 10], 'r--', linewidth=2, label='Perfect Prediction')
        ax.set_xlabel('True Severity')
        ax.set_ylabel('Predicted Severity')
        ax.set_title('Severity Prediction Performance', fontsize=12, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    def _plot_resource_allocation(self, fig):
        """Plot resource allocation distribution"""
        ax = fig.add_subplot(4, 3, 4)
        
        df = self.system.resource_optimizer.create_synthetic_data()
        priority_counts = df['priority'].value_counts()
        
        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8']
        wedges, texts, autotexts = ax.pie(priority_counts, labels=priority_counts.index,
                                           autopct='%1.1f%%', colors=colors,
                                           startangle=90)
        
        for autotext in autotexts:
            autotext.set_color('white')
            autotext.set_fontweight('bold')
        
        ax.set_title('Resource Allocation Distribution', fontsize=12, fontweight='bold')
    
    def _plot_disaster_probabilities(self, fig):
        """Plot disaster type probabilities for sample scenarios"""
        ax = fig.add_subplot(4, 3, 5)
        
        # Create sample scenarios
        scenarios = [
            {'seismic_activity': 7.0, 'temperature': 25, 'humidity': 50, 'wind_speed': 15,
             'rainfall': 10, 'soil_moisture': 45, 'atmospheric_pressure': 1010},
            {'seismic_activity': 1.0, 'temperature': 30, 'humidity': 90, 'wind_speed': 30,
             'rainfall': 150, 'soil_moisture': 90, 'atmospheric_pressure': 990},
            {'seismic_activity': 0.5, 'temperature': 40, 'humidity': 20, 'wind_speed': 40,
             'rainfall': 2, 'soil_moisture': 15, 'atmospheric_pressure': 1015},
        ]
        
        scenario_names = ['High Seismic', 'Heavy Rain', 'Hot & Dry']
        
        probs_data = []
        for scenario in scenarios:
            result = self.system.type_classifier.predict(scenario)
            probs_data.append(list(result['all_probabilities'].values()))
        
        x = np.arange(len(self.system.type_classifier.label_encoder.classes_))
        width = 0.25
        
        for i, (probs, name) in enumerate(zip(probs_data, scenario_names)):
            ax.bar(x + i*width, probs, width, label=name, alpha=0.8)
        
        ax.set_xlabel('Disaster Type')
        ax.set_ylabel('Probability')
        ax.set_title('Disaster Type Probabilities - Sample Scenarios', fontsize=12, fontweight='bold')
        ax.set_xticks(x + width)
        ax.set_xticklabels(self.system.type_classifier.label_encoder.classes_, rotation=45, ha='right')
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
    
    def _plot_severity_heatmap(self, fig):
        """Plot severity vs parameters heatmap"""
        ax = fig.add_subplot(4, 3, 6)
        
        df = self.system.severity_predictor.create_synthetic_data(1000)
        
        # Create correlation matrix
        corr = df.corr()
        
        mask = np.triu(np.ones_like(corr, dtype=bool))
        sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
                   center=0, ax=ax, square=True, linewidths=1)
        ax.set_title('Parameter Correlation Heatmap', fontsize=12, fontweight='bold')
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    
    def _plot_alert_levels(self, fig):
        """Plot alert level distribution"""
        ax = fig.add_subplot(4, 3, 7)
        
        # Generate sample alerts
        severities = np.random.uniform(1, 10, 1000)
        alert_levels = []
        for sev in severities:
            if sev < 3:
                alert_levels.append('GREEN')
            elif sev < 6:
                alert_levels.append('YELLOW')
            elif sev < 8:
                alert_levels.append('ORANGE')
            else:
                alert_levels.append('RED')
        
        alert_counts = pd.Series(alert_levels).value_counts()
        colors_dict = {'GREEN': '#2ECC71', 'YELLOW': '#F1C40F',
                      'ORANGE': '#E67E22', 'RED': '#E74C3C'}
        
        bars = ax.bar(alert_counts.index, alert_counts.values,
                     color=[colors_dict[level] for level in alert_counts.index])
        ax.set_xlabel('Alert Level')
        ax.set_ylabel('Frequency')
        ax.set_title('Alert Level Distribution', fontsize=12, fontweight='bold')
        ax.grid(axis='y', alpha=0.3)
        
        # Add value labels on bars
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{int(height)}', ha='center', va='bottom', fontweight='bold')
    
    def _plot_model_comparison(self, fig):
        """Compare model performance metrics"""
        ax = fig.add_subplot(4, 3, 8)
        
        models = ['Disaster\nClassifier', 'Severity\nPredictor', 'Resource\nOptimizer']
        accuracies = [0.98, 0.85, 0.92]  # Sample metrics
        
        bars = ax.bar(models, accuracies, color=['#3498DB', '#E74C3C', '#2ECC71'])
        ax.set_ylabel('Performance Score')
        ax.set_title('Model Performance Comparison', fontsize=12, fontweight='bold')
        ax.set_ylim([0, 1.1])
        ax.axhline(y=0.9, color='r', linestyle='--', alpha=0.5, label='90% Threshold')
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
        
        # Add value labels
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.2f}', ha='center', va='bottom', fontweight='bold')
    
    def _plot_sensor_patterns(self, fig):
        """Plot sensor data patterns for different disasters"""
        ax = fig.add_subplot(4, 3, 9)
        
        df = self.system.type_classifier.create_synthetic_data(500)
        
        # Normalize data for radar chart representation
        disasters = df['disaster_type'].unique()
        features = ['seismic_activity', 'temperature', 'humidity',
                   'wind_speed', 'rainfall', 'soil_moisture']
        
        for disaster in disasters[:3]:  # Show top 3 for clarity
            disaster_data = df[df['disaster_type'] == disaster][features].mean()
            disaster_norm = (disaster_data - df[features].min()) / (df[features].max() - df[features].min())
            ax.plot(features, disaster_norm, marker='o', label=disaster, linewidth=2)
        
        ax.set_xlabel('Sensor Type')
        ax.set_ylabel('Normalized Reading')
        ax.set_title('Sensor Patterns by Disaster Type', fontsize=12, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    
    def _plot_roc_curves(self, fig):
        """Plot ROC curves for multi-class classification"""
        ax = fig.add_subplot(4, 3, 10)
        
        df = self.system.type_classifier.create_synthetic_data()
        X = df.drop('disaster_type', axis=1)
        y = df['disaster_type']
        
        y_encoded = self.system.type_classifier.label_encoder.transform(y)
        X_scaled = self.system.type_classifier.scaler.transform(X)
        y_proba = self.system.type_classifier.model.predict_proba(X_scaled)
        
        # Binarize labels
        n_classes = len(self.system.type_classifier.label_encoder.classes_)
        y_bin = label_binarize(y_encoded, classes=range(n_classes))
        
        # Compute ROC curve for each class
        for i in range(min(3, n_classes)):  # Show first 3 classes
            fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
            roc_auc = auc(fpr, tpr)
            ax.plot(fpr, tpr, linewidth=2,
                   label=f'{self.system.type_classifier.label_encoder.classes_[i]} (AUC = {roc_auc:.2f})')
        
        ax.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random')
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.set_title('ROC Curves - Multi-class Classification', fontsize=12, fontweight='bold')
        ax.legend(loc='lower right')
        ax.grid(True, alpha=0.3)
    
    def _plot_learning_curves(self, fig):
        """Plot learning curves for model training"""
        ax = fig.add_subplot(4, 3, 11)
        
        # Simulate learning curve data
        train_sizes = np.linspace(0.1, 1.0, 10)
        train_scores = 0.95 - 0.3 * np.exp(-train_sizes * 3) + np.random.normal(0, 0.02, 10)
        val_scores = 0.92 - 0.4 * np.exp(-train_sizes * 3) + np.random.normal(0, 0.03, 10)
        
        ax.plot(train_sizes * 100, train_scores, 'o-', color='blue',
               label='Training Score', linewidth=2, markersize=6)
        ax.plot(train_sizes * 100, val_scores, 'o-', color='red',
               label='Validation Score', linewidth=2, markersize=6)
        
        ax.fill_between(train_sizes * 100, train_scores - 0.02, train_scores + 0.02,
                        alpha=0.1, color='blue')
        ax.fill_between(train_sizes * 100, val_scores - 0.02, val_scores + 0.02,
                        alpha=0.1, color='red')
        
        ax.set_xlabel('Training Set Size (%)')
        ax.set_ylabel('Accuracy Score')
        ax.set_title('Learning Curves', fontsize=12, fontweight='bold')
        ax.legend(loc='lower right')
        ax.grid(True, alpha=0.3)
    
    def _plot_prediction_timeline(self, fig):
        """Plot real-time prediction timeline simulation"""
        ax = fig.add_subplot(4, 3, 12)
        
        # Simulate time series predictions
        times = pd.date_range('2024-01-01 00:00', periods=100, freq='H')
        severities = 3 + 2 * np.sin(np.linspace(0, 4*np.pi, 100)) + np.random.normal(0, 0.5, 100)
        severities = np.clip(severities, 1, 10)
        
        # Color code by alert level
        colors = []
        for sev in severities:
            if sev < 3:
                colors.append('#2ECC71')
            elif sev < 6:
                colors.append('#F1C40F')
            elif sev < 8:
                colors.append('#E67E22')
            else:
                colors.append('#E74C3C')
        
        ax.scatter(times, severities, c=colors, s=30, alpha=0.6)
        ax.plot(times, severities, color='gray', alpha=0.3, linewidth=1)
        
        # Add threshold lines
        ax.axhline(y=3, color='#2ECC71', linestyle='--', alpha=0.5, label='Green')
        ax.axhline(y=6, color='#F1C40F', linestyle='--', alpha=0.5, label='Yellow')
        ax.axhline(y=8, color='#E67E22', linestyle='--', alpha=0.5, label='Orange')
        
        ax.set_xlabel('Time')
        ax.set_ylabel('Severity Score')
        ax.set_title('Disaster Severity Timeline', fontsize=12, fontweight='bold')
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')


# ============= STANDALONE VISUALIZATION FUNCTIONS =============

def generate_individual_plots(system):
    """Generate individual plot files for detailed analysis"""
    
    visualizer = DisasterSystemVisualizer(system)
    
    # 1. Detailed Confusion Matrix
    plt.figure(figsize=(10, 8))
    df = system.type_classifier.create_synthetic_data()
    X = df.drop('disaster_type', axis=1)
    y = df['disaster_type']
    
    y_encoded = system.type_classifier.label_encoder.transform(y)
    X_scaled = system.type_classifier.scaler.transform(X)
    y_pred = system.type_classifier.model.predict(X_scaled)
    
    cm = confusion_matrix(y_encoded, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
               xticklabels=system.type_classifier.label_encoder.classes_,
               yticklabels=system.type_classifier.label_encoder.classes_,
               cbar_kws={'label': 'Count'})
    plt.title('Detailed Confusion Matrix - Disaster Classification',
             fontsize=16, fontweight='bold', pad=20)
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    plt.savefig('confusion_matrix_detailed.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: confusion_matrix_detailed.png")
    plt.close()
    
    # 2. Feature Importance Detailed
    plt.figure(figsize=(12, 6))
    importances = system.type_classifier.model.feature_importances_
    features = ['Seismic Activity', 'Temperature', 'Humidity',
               'Wind Speed', 'Rainfall', 'Soil Moisture', 'Atmospheric Pressure']
    indices = np.argsort(importances)[::-1]
    
    plt.bar(range(len(importances)), importances[indices],
           color=plt.cm.viridis(np.linspace(0, 1, len(importances))))
    plt.xticks(range(len(importances)), [features[i] for i in indices], rotation=45, ha='right')
    plt.xlabel('Features', fontsize=12)
    plt.ylabel('Importance Score', fontsize=12)
    plt.title('Feature Importance Analysis - Disaster Detection',
             fontsize=16, fontweight='bold', pad=20)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('feature_importance_detailed.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: feature_importance_detailed.png")
    plt.close()
    
    # 3. Severity Analysis
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    df_sev = system.severity_predictor.create_synthetic_data(1000)
    X_sev = df_sev.drop('severity', axis=1)
    y_sev = df_sev['severity']
    X_sev_scaled = system.severity_predictor.scaler.transform(X_sev)
    y_pred_sev = system.severity_predictor.model.predict(X_sev_scaled)
    
    # Scatter plot
    axes[0, 0].scatter(y_sev, y_pred_sev, alpha=0.4, s=20, color='steelblue')
    axes[0, 0].plot([1, 10], [1, 10], 'r--', linewidth=2)
    axes[0, 0].set_xlabel('True Severity')
    axes[0, 0].set_ylabel('Predicted Severity')
    axes[0, 0].set_title('Prediction vs Actual')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Residuals
    residuals = y_sev - y_pred_sev
    axes[0, 1].scatter(y_pred_sev, residuals, alpha=0.4, s=20, color='coral')
    axes[0, 1].axhline(y=0, color='r', linestyle='--', linewidth=2)
    axes[0, 1].set_xlabel('Predicted Severity')
    axes[0, 1].set_ylabel('Residuals')
    axes[0, 1].set_title('Residual Analysis')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Distribution
    axes[1, 0].hist(y_sev, bins=30, alpha=0.5, label='True', color='blue')
    axes[1, 0].hist(y_pred_sev, bins=30, alpha=0.5, label='Predicted', color='red')
    axes[1, 0].set_xlabel('Severity')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Severity Distribution')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Error distribution
    axes[1, 1].hist(residuals, bins=30, color='purple', alpha=0.7)
    axes[1, 1].set_xlabel('Prediction Error')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title('Error Distribution')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.suptitle('Severity Prediction Analysis', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig('severity_analysis.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: severity_analysis.png")
    plt.close()
    
    print("\n✓ All individual plots generated successfully!")


# ============= MAIN EXECUTION =============
if __name__ == "__main__":
    # Import the system
    from disaster_alert_system import SmartDisasterAlertSystem
    
    # Initialize and train
    print("Initializing Smart Disaster Alert System...")
    system = SmartDisasterAlertSystem()
    system.train_all_models()
    
    # Create visualizations
    visualizer = DisasterSystemVisualizer(system)
    visualizer.plot_all_visualizations()
    
    # Generate individual detailed plots
    print("\n" + "=" * 60)
    print("GENERATING INDIVIDUAL DETAILED PLOTS")
    print("=" * 60)
    generate_individual_plots(system)
    
    print("\n" + "=" * 60)
    print("✓ ALL VISUALIZATIONS COMPLETED!")
    print("=" * 60)
    print("\nGenerated Files:")
    print("  1. disaster_system_analysis.png (12-panel comprehensive view)")
    print("  2. confusion_matrix_detailed.png")
    print("  3. feature_importance_detailed.png")
    print("  4. severity_analysis.png (4-panel severity analysis)")